#### Workflow Objectives: 

The primary focus of this phase is to ensure high-quality raw data and accurate quantification against the reference transcriptome. The workflow utilizes four core stages:

* **Initial QC**: Perform FastQC (v0.12.1) to evaluate per-base sequence quality, GC content, sequence duplication, and adapter contamination.
* **Contamination Screening**: Execute FastQ-Screen (v0.14.1) to identify and filter cross-species contaminant RNAs.
* **Read Trimming**: Utilize Trimmomatic (v0.39) to remove residual adapters and low-quality bases.
* **Quantification**: Pseudo-align processed reads to the human reference transcriptome using Kallisto (v0.46.0).



#### Data Processing & Implementation
All RNA-seq processing tasks (FastQC, trimming, alignment, quantification) are submitted to the CHOP HPC cluster using the **SLURM scheduler** (default: 4 GB memory, 1 CPU, 6-hour runtime). Each step is executed as a separate job script to ensure efficient resource allocation and reproducibility. 

#### Step 1: Pre-alignment Quality Control
Initial FastQC and MultiQC runs evaluate the integrity of all raw FASTQ files.

In [ ]:
#!/bin/sh

# Script to run FastQC on the raw FASTQ files

# Assumes:
#  - FASTQ files are in ./fastq_files
#  - FastQC is installed at ~/tools/FastQC/fastqc (edit if different)
#  - multiqc is available in the conda environment

# Create output directory
mkdir -p fastqc_results

# Run FastQC
~/tools/FastQC/fastqc fastq_files/*.fastq.gz -o fastqc_results

# Run MultiQC to aggregate results
conda activate python3.7
multiqc fastqc_results/ -o fastqc_results/.
conda deactivate

**1.1 Mean Quality Scores**   
All base quality scores were within acceptable ranges.  
<img src="https://www.dropbox.com/scl/fi/0x9w8g4a6ka7bygibp2pu/fastqc_per_base_sequence_quality_plot.png?rlkey=obdi0kxvaepxtmn34o2z5pn21&st=gr5hd5he&raw=1" width="450">

**1.2 Per Sequence GC Content**   
With RNA sequencing there may be a greater or lesser distribution of mean GC content among transcripts causing the observed plot to be wider or narrower than an ideal normal distribution. Sharp peaks on an otherwise smooth distribution are normally the result of a specific contaminant (adapter dimers for example).  
<img src="https://www.dropbox.com/scl/fi/go3vkxry7mhrjt23wktxh/fastqc_per_sequence_gc_content_plot.png?rlkey=9nj3p7jamu9lz8f01rb6xcq4f&st=majtu7o8&raw=1" width="450">

**1.3 Sequence Duplication**   
Truly over-represented sequences such as very abundant transcripts in an RNA-Seq library.  
<img src="https://www.dropbox.com/scl/fi/2sj0i4fer2y55fw1q3iqv/fastqc_sequence_duplication_levels_plot.png?rlkey=ueg2v83s0ecwr6242e5lq9xld&st=crsejrz1&raw=1" width="450">

**1.4 Adapter Content**   
Residual Illumina Universal Adapter sequences were detected.  
<img src="https://www.dropbox.com/scl/fi/m8vtr3m7r4tlv4rpczus4/fastqc_adapter_content_plot.png?rlkey=jfz0tey15jofjs3s6gqbp36z3&st=b364cbe3&raw=1" width="450">

#### Step 2: Identify contamination in FASTQ files  
Screens reads against multiple reference genomes to ensure sample purity.

In [ ]:
#!/bin/sh
# Script for identifying and filtering contaminant RNAs

# Assumes:
#  - FASTQ files are in ./fastq_files and end in .fastq.gz
#  - FastQ-Screen is installed at ~/tools/FastQ-Screen-0.14.1/fastq_screen (edit if different)
#  - A valid configuration file exists at ./config/fastq_screen.conf
#  - The module system has Bowtie2/2.3.4.2-foss-2018b available to load
#  - Conda is initialized, and an environment named 'python3.7' contains multiqc

module load Bowtie2/2.3.4.2-foss-2018b
mkdir -p fastq_screen_results

for f in $(ls fastq_files/*fastq.gz)
do
    ~/tools/FastQ-Screen-0.14.1/fastq_screen --conf ./config/fastq_screen.conf ${f} --outdir fastq_screen_results
done

conda activate python3.7  
multiqc fastq_screen_results/ -o fastq_screen_results/  
conda deactivate

No obvious contamination found. Hits in human, mouse, and rat genomes are consistent with repetitive sequences or homologous regions, reflecting expected cross-species similarity rather than true contamination.  

<img src="https://www.dropbox.com/scl/fi/x80d0o1cqptea26pzjmcp/fastq_screen_plot-edited.png?rlkey=5huqkfpshj81xpuemupo4k6l5&st=mv02q62p&raw=1" width="458">  
<img src="https://www.dropbox.com/scl/fi/xmhppt85w4al0o88r09h7/fastq_screen.png?rlkey=3griugclvt3ajsi7qx3k629yg&st=lxoaa3sk&raw=1" width="500">

#### Step 3: Read trimming

Removes adapters and filters low-quality sequences to prepare reads for quantification.

In [ ]:
#!/bin/sh
# Script for filtering low quality reads and adapter contamination

#SBATCH --mem=8G 
#SBATCH -t 24:00:00

# Assumes:
#  - This script is submitted to a SLURM workload manager (due to #SBATCH directives).
#  - Paired-end FASTQ files are located in ./fastq_files/ and follow the naming convention *_R1_001.fastq.gz and *_R2_001.fastq.gz.
#  - The module system has Trimmomatic/0.39-Java-11 available, and loading it correctly sets the $EBROOTTRIMMOMATIC environment variable.
#  - Conda is initialized, and an environment named 'python3.7' contains multiqc.

module load Trimmomatic/0.39-Java-11
mkdir -p trimmomatic_results

# Process paired-end reads
for f in $(ls fastq_files/*.fastq.gz | sed 's/.*fastq_files\///' | sed 's/_R.*//' | sort -u) 
do 
    java -jar $EBROOTTRIMMOMATIC/trimmomatic-0.39.jar PE -phred33 \
    fastq_files/${f}_R1_001.fastq.gz fastq_files/${f}_R2_001.fastq.gz \
    trimmomatic_results/${f}_R1_001_paired.fastq.gz trimmomatic_results/${f}_R1_001_unpaired.fastq.gz \
    trimmomatic_results/${f}_R2_001_paired.fastq.gz trimmomatic_results/${f}_R2_001_unpaired.fastq.gz \
    ILLUMINACLIP:$EBROOTTRIMMOMATIC/adapters/TruSeq3-PE-2.fa:2:30:10 LEADING:30 TRAILING:30 SLIDINGWINDOW:4:20 MINLEN:70 
done

# Run FastQC on trimmed data
mkdir -p ./trimmomatic_results/fastqc_results
~/tools/FastQC/fastqc ./trimmomatic_results/*_paired.fastq.gz -o ./trimmomatic_results/fastqc_results

# Aggregate trimmed reports
conda activate python3.7
multiqc ./trimmomatic_results/fastqc_results/*_paired_fastqc.zip -o ./trimmomatic_results/fastqc_results/
conda deactivate

#### Step 4: Pseudo-alignment 

Quantifies transcript abundances directly against the human reference transcriptome.

In [ ]:
#!/bin/sh
# Script for pseudo-alignment and quantification

# Assumes:
#  - Conda is initialized, and an environment named 'python3.7' contains both kallisto and multiqc.
#  - Trimmed paired-end FASTQ files are located in ./trimmomatic_results/ and follow the naming convention *_R1_001_paired.fastq.gz and *_R2_001_paired.fastq.gz.
#  - A pre-built Kallisto index for the human transcriptome exists at ./reference/Homo_sapiens.GRCh38.cdna.all.index.
#  - The script is executed via a SLURM workload manager (e.g., using sbatch), which generates slurm-*.out log files in the current working directory for MultiQC to parse.

conda activate python3.7
mkdir -p kallisto_results

for f in $(ls ./trimmomatic_results/*_paired.fastq.gz | sed 's/.*_results\///' | sed 's/_R.*//' | sort -u)
do
    kallisto quant -i ./reference/Homo_sapiens.GRCh38.cdna.all.index \
    -o ./kallisto_results/${f} \
    ./trimmomatic_results/${f}_R1_001_paired.fastq.gz ./trimmomatic_results/${f}_R2_001_paired.fastq.gz
done

multiqc ./slurm-*.out -p -o ./kallisto_results/.
conda deactivate

All samples show high alignment rates (90–93%) with fragment lengths mostly between 255–311 bp, indicating robust Kallisto quantification.  

<img src="https://www.dropbox.com/scl/fi/s0kv37hwkxhv6m559capj/kallisto_alignment-edited.png?rlkey=tnveodqaqktrozp7lmx9vhabk&st=cz7sux18&raw=1" width=500>




### Next Steps
Low-abundance genes will be filtered, and expression values normalized. Sample clustering and variability will be assessed via PCA, and differential expression will be analyzed. Functional enrichment will be performed using GSEA through clusterProfiler 4.4.4 with MSigDB gene sets. Data visualization will include heatmaps (gplots 3.1.3), network diagrams (Cytoscape 3.9.1), and plots generated with ggplot2 3.4.0.